In [12]:
import os
from datetime import datetime

import pandas as pd
from feast import FeatureStore

In [32]:
raw_data_path = os.path.join("feature_store", "feature_repo", "data", "driver_stats.parquet")
feature_store_path = os.path.join("feature_store", "feature_repo")

In [44]:
df = pd.read_parquet(raw_data_path)

In [45]:
df.head(5)

,event_timestamp,driver_id,conv_rate,acc_rate,avg_daily_trips,created
0,2024-10-17 12:07:08.228578+00:00,1001,1.000000,1.000000,1000,2024-10-17 12:07:08.228581
1,2024-10-02 11:00:00+00:00,1005,0.429879,0.194598,582,2024-10-17 11:30:07.072000
2,2024-10-02 12:00:00+00:00,1005,0.230119,0.642878,551,2024-10-17 11:30:07.072000
3,2024-10-02 13:00:00+00:00,1005,0.128600,0.674187,38,2024-10-17 11:30:07.072000
4,2024-10-02 14:00:00+00:00,1005,0.400603,0.473636,583,2024-10-17 11:30:07.072000


In [46]:
store = FeatureStore(repo_path=feature_store_path)
entity_df = df[["driver_id", "event_timestamp"]].copy()

realtime_driver_metrics использует 2 созданных FeatureView

1. FeatureView - как хорошо водитель выполняет работу

2. FeatureView  - активность водителя

In [47]:
training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_quality_stats:conv_rate",
        "driver_quality_stats:acc_rate",
        "driver_activity_stats:avg_daily_trips",
        "realtime_driver_metrics:quality_index",
        "realtime_driver_metrics:efficiency_score",
    ],
).to_df()

In [48]:
training_df.head()

,driver_id,event_timestamp,conv_rate,acc_rate,avg_daily_trips,quality_index,efficiency_score
0,1003,2021-04-12 07:00:00+00:00,0.697411,0.197680,25,0.547492,1.783782
1,1004,2021-04-12 07:00:00+00:00,0.774095,0.929545,796,0.820730,5.483178
2,1005,2021-04-12 07:00:00+00:00,0.186924,0.576559,648,0.303815,1.967331
3,1001,2021-04-12 07:00:00+00:00,0.709758,0.692957,402,0.704718,4.227557
4,1002,2021-04-12 07:00:00+00:00,0.718295,0.584081,370,0.678031,4.011369


Получение признаков для модели в реальном времени 

In [49]:
online_features = store.get_online_features(
    features=[
        "driver_quality_stats:conv_rate",
        "driver_activity_stats:avg_daily_trips",
        "realtime_driver_metrics:efficiency_score",
    ],
    entity_rows=[
        {"driver_id": 1001},
        {"driver_id": 1005},
    ],
).to_dict()

/home/suv/.cache/pypoetry/virtualenvs/feature-store-0pgsfV9a-py3.12/lib/python3.12/site-packages/feast/infra/online_stores/sqlite.py:220: DeprecationWarning: The default timestamp converter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  rows = cur.fetchall()


In [50]:
df["event_timestamp"].min(), df["event_timestamp"].max()

(Timestamp('2021-04-12 07:00:00+0000', tz='UTC'),
 Timestamp('2024-10-17 12:07:08.228578+0000', tz='UTC'))

In [51]:
print(online_features)

{'driver_id': [1001, 1005], 'conv_rate': [1.0, 0.5300146341323853], 'avg_daily_trips': [1000, 228], 'efficiency_score': [6.90875477931522, 3.2420150999045743]}
